In [ ]:
!pip install pyspark

In [4]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
from pathlib import Path
spark = SparkSession.builder.appName('practice_set2').getOrCreate()

In [3]:
from google.colab import files

uploaded = files.upload()

Saving shipments_75000.csv to shipments_75000.csv
Saving shipments_extra_12000.csv to shipments_extra_12000.csv


In [5]:
shipments_df = spark.read.option("header", True).option("inferSchema", True).csv("shipments_75000.csv")
extra_df = spark.read.option("header", True).option("inferSchema", True).csv("shipments_extra_12000.csv")
warehouse_df = spark.read.option("header", True).option("inferSchema", True).csv("warehouse_master.csv")
carrier_df = spark.read.option("header", True).option("inferSchema", True).csv("carrier_master.csv")
route_df = spark.read.option("header", True).option("inferSchema", True).csv("route_master.csv")
sensor_df = spark.read.option("multiline", True).json("warehouse_sensor_config.json")


In [6]:
shipments_df.show(5, truncate=False)
warehouse_df.show(5, truncate=False)
carrier_df.show(5, truncate=False)
route_df.show(5, truncate=False)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|payment_mode    |priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|SHP0000001 |2026-05-24  |2026-05-24   |2026-05-26            |2

Part A

In [7]:
shipments_df.printSchema()
print("Total shipments:", shipments_df.count())
shipments_df.show(15, truncate=False)

root
 |-- shipment_id: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_weight_kg: double (nullable = true)
 |-- shipping_fee_inr: double (nullable = true)
 |-- shipping_cost_inr: double (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- delivery_attempts: integer (nullable = true)
 |-- damage_flag: string (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- handling_minutes: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- priority_flag: string (nullable = true)

Total shipments: 75000
+-----------+------------+-------------+------------

In [9]:
exercise_2_df = shipments_df.select("shipment_id", "warehouse_id", "service_level", "shipment_status", "shipping_fee_inr")
exercise_2_df.show(10, truncate=False)

+-----------+------------+-------------+---------------+----------------+
|shipment_id|warehouse_id|service_level|shipment_status|shipping_fee_inr|
+-----------+------------+-------------+---------------+----------------+
|SHP0000001 |W08         |Express      |Delivered      |554.34          |
|SHP0000002 |W08         |Standard     |Delivered      |414.56          |
|SHP0000003 |W01         |Express      |Delivered      |724.64          |
|SHP0000004 |W03         |Standard     |Delivered      |287.86          |
|SHP0000005 |W03         |Economy      |Delivered      |251.68          |
|SHP0000006 |W01         |Express      |Delivered      |477.9           |
|SHP0000007 |W02         |Standard     |In Transit     |298.68          |
|SHP0000008 |W03         |Economy      |In Transit     |284.52          |
|SHP0000009 |W02         |Standard     |Delivered      |861.74          |
|SHP0000010 |W02         |Economy      |Delivered      |309.76          |
+-----------+------------+------------

In [10]:
exercise_3_df = shipments_df.filter((F.col("service_level") == "Express") & (F.col("package_weight_kg") > 5))
exercise_3_df.show(20, truncate=False)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|payment_mode    |priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|SHP0000003 |2026-01-31  |2026-02-01   |2026-02-03            |2

In [11]:
exercise_4_df = shipments_df.orderBy(F.col("shipping_fee_inr").desc())
exercise_4_df.show(20, truncate=False)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|payment_mode    |priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|SHP0044904 |2026-08-11  |2026-08-13   |2026-08-16            |N

Part B

In [13]:
exercise_5_df = shipments_df.groupBy("shipment_status").agg(F.count("*").alias("shipment_count")).orderBy(F.desc("shipment_count"))
exercise_5_df.show(truncate=False)

+---------------+--------------+
|shipment_status|shipment_count|
+---------------+--------------+
|Delivered      |52434         |
|In Transit     |10472         |
|Returned       |6130          |
|Cancelled      |3723          |
|Lost           |2241          |
+---------------+--------------+



In [14]:
exercise_6_df = shipments_df.groupBy("service_level").agg(
    F.sum("shipping_fee_inr").alias("total_shipping_fees_inr"),
    F.sum(F.coalesce(F.col("shipping_cost_inr"), F.lit(0))).alias("total_shipping_costs_inr")
).withColumn("profit_inr", F.col("total_shipping_fees_inr") - F.col("total_shipping_costs_inr")).withColumn(
    "profit_margin_pct", F.when(F.col("total_shipping_fees_inr") != 0, F.col("profit_inr") / F.col("total_shipping_fees_inr") * 100)
)
exercise_6_df.show(truncate=False)

+-------------+-----------------------+------------------------+------------------+------------------+
|service_level|total_shipping_fees_inr|total_shipping_costs_inr|profit_inr        |profit_margin_pct |
+-------------+-----------------------+------------------------+------------------+------------------+
|Express      |1.0058468059999986E7   |7350805.699999983       |2707662.360000002 |26.919232072403737|
|Standard     |1.3437372619999977E7   |1.0110465900000025E7    |3326906.7199999522|24.758610288504137|
|Economy      |4489918.680000002      |3386212.500000007       |1103706.1799999946|24.581874609809063|
+-------------+-----------------------+------------------------+------------------+------------------+



In [15]:
exercise_7_df = shipments_df.join(warehouse_df, "warehouse_id", "left").groupBy("region").agg(F.count("*").alias("shipment_count")).orderBy(F.desc("shipment_count"))
exercise_7_df.show(truncate=False)

+------+--------------+
|region|shipment_count|
+------+--------------+
|South |33018         |
|West  |26909         |
|North |9767          |
|East  |5306          |
+------+--------------+



In [17]:
shipment_route_df = shipments_df.join(carrier_df, "carrier_id", "left").join(route_df, "route_id", "left")
exercise_8_df = shipment_route_df.filter(
    F.col("shipping_cost_inr").isNotNull() & F.col("distance_km").isNotNull() & (F.col("distance_km") != 0)
).withColumn("cost_per_km", F.col("shipping_cost_inr") / F.col("distance_km")).groupBy("carrier_id").agg(
    F.avg("cost_per_km").alias("average_shipping_cost_per_km")
).orderBy("carrier_id")
exercise_8_df.show(truncate=False)

+----------+----------------------------+
|carrier_id|average_shipping_cost_per_km|
+----------+----------------------------+
|NULL      |0.6318401277501529          |
|C01       |0.6032748764541479          |
|C02       |0.5957409571811839          |
|C03       |0.594275960971091           |
|C04       |0.6125755103088726          |
|C05       |0.5897283901696387          |
|C06       |0.5879324060720517          |
|C07       |0.6074832283099014          |
+----------+----------------------------+



Part C

In [18]:
exercise_9_df = shipments_df.agg(
    F.sum(F.col("carrier_id").isNull().cast("long")).alias("missing_carrier_id"),
    F.sum(F.col("package_weight_kg").isNull().cast("long")).alias("missing_package_weight_kg"),
    F.sum(F.col("shipping_cost_inr").isNull().cast("long")).alias("missing_shipping_cost_inr"),
    F.sum(F.col("customer_rating").isNull().cast("long")).alias("missing_customer_rating")
)
exercise_9_df.show()

+------------------+-------------------------+-------------------------+-----------------------+
|missing_carrier_id|missing_package_weight_kg|missing_shipping_cost_inr|missing_customer_rating|
+------------------+-------------------------+-------------------------+-----------------------+
|               871|                     1129|                     1503|                  26255|
+------------------+-------------------------+-------------------------+-----------------------+



In [19]:
exercise_10a_df = shipments_df.na.fill({"handling_minutes": 0})
exercise_10a_df.select("shipment_id", "handling_minutes").show(10)

+-----------+----------------+
|shipment_id|handling_minutes|
+-----------+----------------+
| SHP0000001|           122.0|
| SHP0000002|           220.0|
| SHP0000003|           188.0|
| SHP0000004|           179.0|
| SHP0000005|           218.0|
| SHP0000006|            48.0|
| SHP0000007|           214.0|
| SHP0000008|            76.0|
| SHP0000009|           222.0|
| SHP0000010|            98.0|
+-----------+----------------+
only showing top 10 rows


In [20]:
exercise_10b_df = shipments_df.na.drop(subset=["carrier_id"])
print("Before:", shipments_df.count())
print("After:", exercise_10b_df.count())

Before: 75000
After: 74129


In [21]:
exercise_11_df = shipments_df.withColumn("weight_category",
    F.when(F.col("package_weight_kg").isNull(), "Unknown")
     .when(F.col("package_weight_kg") < 2, "Light")
     .when(F.col("package_weight_kg") <= 10, "Medium").otherwise("Heavy"))
exercise_11_df.select("shipment_id", "package_weight_kg", "weight_category").show(20, truncate=False)

+-----------+-----------------+---------------+
|shipment_id|package_weight_kg|weight_category|
+-----------+-----------------+---------------+
|SHP0000001 |3.93             |Medium         |
|SHP0000002 |5.62             |Medium         |
|SHP0000003 |8.78             |Medium         |
|SHP0000004 |0.97             |Light          |
|SHP0000005 |NULL             |Unknown        |
|SHP0000006 |6.05             |Medium         |
|SHP0000007 |1.36             |Light          |
|SHP0000008 |2.04             |Medium         |
|SHP0000009 |18.73            |Heavy          |
|SHP0000010 |2.02             |Medium         |
|SHP0000011 |6.87             |Medium         |
|SHP0000012 |11.83            |Heavy          |
|SHP0000013 |3.73             |Medium         |
|SHP0000014 |1.42             |Light          |
|SHP0000015 |1.74             |Light          |
|SHP0000016 |6.87             |Medium         |
|SHP0000017 |2.58             |Medium         |
|SHP0000018 |1.85             |Light    

Part D

In [22]:
exercise_12_df = shipments_df
for c in ["booking_date", "promised_delivery_date", "actual_delivery_date"]:
    exercise_12_df = exercise_12_df.withColumn(c, F.to_date(F.col(c)))
exercise_12_df = exercise_12_df.withColumn("delivery_duration_days",
    F.when((F.col("shipment_status") == "Delivered") & F.col("actual_delivery_date").isNotNull(),
           F.datediff("actual_delivery_date", "booking_date")))
exercise_12_df.select("shipment_id", "booking_date", "actual_delivery_date", "delivery_duration_days").show(20, truncate=False)

+-----------+------------+--------------------+----------------------+
|shipment_id|booking_date|actual_delivery_date|delivery_duration_days|
+-----------+------------+--------------------+----------------------+
|SHP0000001 |2026-05-24  |2026-05-27          |3                     |
|SHP0000002 |2026-05-17  |2026-05-21          |4                     |
|SHP0000003 |2026-01-31  |2026-02-04          |4                     |
|SHP0000004 |2026-03-12  |2026-03-18          |6                     |
|SHP0000005 |2026-08-16  |2026-08-29          |13                    |
|SHP0000006 |2026-02-06  |2026-02-09          |3                     |
|SHP0000007 |2026-01-10  |NULL                |NULL                  |
|SHP0000008 |2026-03-06  |NULL                |NULL                  |
|SHP0000009 |2026-08-12  |2026-08-21          |9                     |
|SHP0000010 |2026-06-01  |2026-06-11          |10                    |
|SHP0000011 |2026-04-24  |2026-05-01          |7                     |
|SHP00

In [23]:
exercise_13_df = exercise_12_df.withColumn("delivery_performance",
    F.when(F.col("actual_delivery_date").isNull(), "Missing Actual Date")
     .when(F.col("actual_delivery_date") < F.col("promised_delivery_date"), "Early")
     .when(F.col("actual_delivery_date") == F.col("promised_delivery_date"), "On Time")
     .otherwise("Late"))
exercise_13_df.select("shipment_id", "promised_delivery_date", "actual_delivery_date", "delivery_performance").show(20, truncate=False)

+-----------+----------------------+--------------------+--------------------+
|shipment_id|promised_delivery_date|actual_delivery_date|delivery_performance|
+-----------+----------------------+--------------------+--------------------+
|SHP0000001 |2026-05-26            |2026-05-27          |Late                |
|SHP0000002 |2026-05-21            |2026-05-21          |On Time             |
|SHP0000003 |2026-02-03            |2026-02-04          |Late                |
|SHP0000004 |2026-03-17            |2026-03-18          |Late                |
|SHP0000005 |2026-08-25            |2026-08-29          |Late                |
|SHP0000006 |2026-02-09            |2026-02-09          |On Time             |
|SHP0000007 |2026-01-14            |NULL                |Missing Actual Date |
|SHP0000008 |2026-03-16            |NULL                |Missing Actual Date |
|SHP0000009 |2026-08-18            |2026-08-21          |Late                |
|SHP0000010 |2026-06-10            |2026-06-11      

In [24]:
exercise_14_df = exercise_12_df.filter(F.col("shipment_status") == "Delivered").withColumn(
    "booking_month", F.date_format("booking_date", "yyyy-MM")
).groupBy("booking_month").agg(
    F.count("*").alias("delivered_shipment_count"),
    F.sum("shipping_fee_inr").alias("total_shipping_fees_inr")
).orderBy("booking_month")
exercise_14_df.show(truncate=False)

+-------------+------------------------+-----------------------+
|booking_month|delivered_shipment_count|total_shipping_fees_inr|
+-------------+------------------------+-----------------------+
|2026-01      |6710                    |2525138.440000005      |
|2026-02      |6097                    |2268113.5000000037     |
|2026-03      |6810                    |2530456.2400000026     |
|2026-04      |6379                    |2389163.4600000014     |
|2026-05      |6858                    |2549683.9999999963     |
|2026-06      |6615                    |2447590.1200000006     |
|2026-07      |6773                    |2512355.2199999997     |
|2026-08      |6192                    |2306432.5199999996     |
+-------------+------------------------+-----------------------+



Part E

In [25]:
carrier_region_fees_df = shipments_df.join(warehouse_df.select("warehouse_id", "region"), "warehouse_id", "left").groupBy("region", "carrier_id").agg(F.sum("shipping_fee_inr").alias("total_shipping_fees_inr"))
w = Window.partitionBy("region").orderBy(F.desc("total_shipping_fees_inr"))
exercise_15_df = carrier_region_fees_df.withColumn("carrier_rank", F.dense_rank().over(w)).orderBy("region", "carrier_rank")
exercise_15_df.show(truncate=False)

+------+----------+-----------------------+------------+
|region|carrier_id|total_shipping_fees_inr|carrier_rank|
+------+----------+-----------------------+------------+
|East  |C01       |393862.61999999994     |1           |
|East  |C02       |373590.02              |2           |
|East  |C03       |302260.92000000004     |3           |
|East  |C05       |243322.91999999998     |4           |
|East  |C07       |222025.85999999993     |5           |
|East  |C04       |211583.64000000004     |6           |
|East  |C06       |195810.3399999999      |7           |
|East  |NULL      |24594.6                |8           |
|North |C02       |731442.0999999994      |1           |
|North |C01       |653223.6200000005      |2           |
|North |C03       |597883.4600000005      |3           |
|North |C05       |464601.85999999975     |4           |
|North |C07       |425396.69999999995     |5           |
|North |C04       |393181.07999999984     |6           |
|North |C06       |338296.45999

In [26]:
w = Window.partitionBy("warehouse_id").orderBy(F.col("booking_date").desc_nulls_last(), F.col("shipment_id").desc())
exercise_16_df = shipments_df.withColumn("row_num", F.row_number().over(w)).filter(F.col("row_num") == 1).drop("row_num")
exercise_16_df.show(truncate=False)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|payment_mode    |priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|SHP0072346 |2026-08-28  |NULL         |2026-09-03            |N

In [27]:
w = Window.partitionBy("warehouse_id").orderBy("booking_date", "shipment_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
exercise_17_df = shipments_df.withColumn("running_shipping_fee_inr", F.sum("shipping_fee_inr").over(w))
exercise_17_df.select("warehouse_id", "booking_date", "shipment_id", "shipping_fee_inr", "running_shipping_fee_inr").show(20, truncate=False)

+------------+------------+-----------+----------------+------------------------+
|warehouse_id|booking_date|shipment_id|shipping_fee_inr|running_shipping_fee_inr|
+------------+------------+-----------+----------------+------------------------+
|W01         |2026-01-01  |SHP0004062 |312.48          |312.48                  |
|W01         |2026-01-01  |SHP0006739 |227.02          |539.5                   |
|W01         |2026-01-01  |SHP0007553 |316.02          |855.52                  |
|W01         |2026-01-01  |SHP0008505 |572.68          |1428.1999999999998      |
|W01         |2026-01-01  |SHP0009674 |322.9           |1751.1                  |
|W01         |2026-01-01  |SHP0011312 |390.1           |2141.2                  |
|W01         |2026-01-01  |SHP0011588 |421.72          |2562.92                 |
|W01         |2026-01-01  |SHP0011698 |190.22          |2753.14                 |
|W01         |2026-01-01  |SHP0011973 |384.64          |3137.7799999999997      |
|W01         |20

Part F

In [28]:
combined_df = shipments_df.unionByName(extra_df, allowMissingColumns=True)
print("Original:", shipments_df.count(), "Extra:", extra_df.count(), "Combined:", combined_df.count())

Original: 75000 Extra: 12000 Combined: 87000


In [29]:
repeated_ids_df = combined_df.groupBy("shipment_id").count().filter(F.col("count") > 1)
repeated_ids_df.show(truncate=False)

+-----------+-----+
|shipment_id|count|
+-----------+-----+
|SHP0039848 |2    |
|SHP0048411 |2    |
|SHP0052790 |2    |
|SHP0026413 |2    |
|SHP0041611 |2    |
|SHP0010090 |2    |
|SHP0038340 |2    |
|SHP0027530 |2    |
|SHP0031755 |2    |
|SHP0034259 |2    |
|SHP0047068 |2    |
|SHP0048280 |2    |
|SHP0008086 |2    |
|SHP0031960 |2    |
|SHP0025784 |2    |
|SHP0005611 |2    |
|SHP0011388 |2    |
|SHP0017357 |2    |
|SHP0035268 |2    |
|SHP0002969 |2    |
+-----------+-----+
only showing top 20 rows


In [30]:
deduplicated_df = combined_df.dropDuplicates(["shipment_id"])
print("Before:", combined_df.count())
print("After:", deduplicated_df.count())

Before: 87000
After: 86250


In [31]:
sensor_df.printSchema()
sensor_df.show(truncate=False)

root
 |-- environment: struct (nullable = true)
 |    |-- humidity_target_pct: long (nullable = true)
 |    |-- temperature_range_c: struct (nullable = true)
 |    |    |-- maximum: long (nullable = true)
 |    |    |-- minimum: long (nullable = true)
 |-- safety: struct (nullable = true)
 |    |-- fire_suppression: string (nullable = true)
 |    |-- inspections: struct (nullable = true)
 |    |    |-- last_date: string (nullable = true)
 |    |    |-- next_date: string (nullable = true)
 |-- sensors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- enabled: boolean (nullable = true)
 |    |    |-- sensor_id: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |-- warehouse_id: string (nullable = true)

+--------------+-------------------------------------+------------------------------------------------------------------------------+------------+
|environment   |safety                               |sensors                     

In [33]:
exercise_20_df = sensor_df.select(
    F.col("warehouse_id"),
    F.col("environment.temperature_range_c.minimum").alias("min_temperature_c"),
    F.col("environment.temperature_range_c.maximum").alias("max_temperature_c"),
    F.col("environment.humidity_target_pct").alias("humidity_target_pct"),
    F.explode_outer("sensors").alias("sensor")
).select(
    "warehouse_id",
    "min_temperature_c",
    "max_temperature_c",
    "humidity_target_pct",
    F.col("sensor.sensor_id").alias("sensor_id"),
    F.col("sensor.type").alias("sensor_type"),
    F.col("sensor.enabled").alias("enabled")
)

exercise_20_df.show(30, truncate=False)

+------------+-----------------+-----------------+-------------------+---------+-----------+-------+
|warehouse_id|min_temperature_c|max_temperature_c|humidity_target_pct|sensor_id|sensor_type|enabled|
+------------+-----------------+-----------------+-------------------+---------+-----------+-------+
|W01         |18               |32               |45                 |W01-S1   |Temperature|false  |
|W01         |18               |32               |45                 |W01-S2   |Humidity   |true   |
|W01         |18               |32               |45                 |W01-S3   |Door       |true   |
|W02         |18               |32               |50                 |W02-S1   |Temperature|true   |
|W02         |18               |32               |50                 |W02-S2   |Humidity   |true   |
|W02         |18               |32               |50                 |W02-S3   |Door       |true   |
|W03         |18               |32               |55                 |W03-S1   |Temperature

Part G

In [34]:
exercise_21_df = shipments_df.filter(F.col("shipment_status") == "Delivered").join(
    warehouse_df.select("warehouse_id", "region"), "warehouse_id", "left"
).withColumn("estimated_profit_inr", F.col("shipping_fee_inr") - F.coalesce(F.col("shipping_cost_inr"), F.lit(0))).groupBy("region").agg(
    F.sum("estimated_profit_inr").alias("total_profit_inr"), F.count("*").alias("shipment_count")
).orderBy(F.desc("total_profit_inr"))
exercise_21_df.explain()
exercise_21_df.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_profit_inr#1513 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_profit_inr#1513 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2019]
      +- HashAggregate(keys=[region#94], functions=[sum(estimated_profit_inr#1512), count(1)])
         +- Exchange hashpartitioning(region#94, 200), ENSURE_REQUIREMENTS, [plan_id=2016]
            +- HashAggregate(keys=[region#94], functions=[partial_sum(estimated_profit_inr#1512), partial_count(1)])
               +- Project [region#94, (shipping_fee_inr#28 - coalesce(shipping_cost_inr#29, 0.0)) AS estimated_profit_inr#1512]
                  +- BroadcastHashJoin [warehouse_id#22], [warehouse_id#91], LeftOuter, BuildRight, false
                     :- Project [warehouse_id#22, shipping_fee_inr#28, shipping_cost_inr#29]
                     :  +- Filter (isnotnull(shipment_status#30) AND (shipment_status#30 = Delivered))
                     :     +- FileScan c

In [35]:
cached_delivered_df = shipments_df.filter(F.col("shipment_status") == "Delivered").cache()
t = time.time(); count_1 = cached_delivered_df.count(); time_1 = time.time() - t
t = time.time(); count_2 = cached_delivered_df.count(); time_2 = time.time() - t
t = time.time(); fees_1 = cached_delivered_df.agg(F.sum("shipping_fee_inr")).first()[0]; time_3 = time.time() - t
t = time.time(); fees_2 = cached_delivered_df.agg(F.sum("shipping_fee_inr")).first()[0]; time_4 = time.time() - t
print("Counts:", count_1, count_2, "times:", round(time_1, 3), round(time_2, 3))
print("Fee totals:", fees_1, fees_2, "times:", round(time_3, 3), round(time_4, 3))
cached_delivered_df.unpersist()

Counts: 52434 52434 times: 2.162 0.11
Fee totals: 19528933.499999907 19528933.499999907 times: 0.257 0.127


DataFrame[shipment_id: string, booking_date: date, dispatch_date: date, promised_delivery_date: date, actual_delivery_date: date, warehouse_id: string, carrier_id: string, route_id: string, customer_segment: string, service_level: string, package_weight_kg: double, shipping_fee_inr: double, shipping_cost_inr: double, shipment_status: string, delivery_attempts: int, damage_flag: string, customer_rating: double, handling_minutes: double, payment_mode: string, priority_flag: string]

In [36]:
express_persisted_df = shipments_df.filter(F.col("service_level") == "Express").persist(StorageLevel.MEMORY_AND_DISK)
print("Count:", express_persisted_df.count())
print("Sample size:", len(express_persisted_df.take(10)))
print("Storage level:", express_persisted_df.storageLevel)
express_persisted_df.unpersist()

Count: 21090
Sample size: 10
Storage level: Disk Memory Serialized 1x Replicated


DataFrame[shipment_id: string, booking_date: date, dispatch_date: date, promised_delivery_date: date, actual_delivery_date: date, warehouse_id: string, carrier_id: string, route_id: string, customer_segment: string, service_level: string, package_weight_kg: double, shipping_fee_inr: double, shipping_cost_inr: double, shipment_status: string, delivery_attempts: int, damage_flag: string, customer_rating: double, handling_minutes: double, payment_mode: string, priority_flag: string]

Part H

In [37]:
exercise_24_df = shipments_df.repartition(8).withColumn("partition_id", F.spark_partition_id())
exercise_24_df.groupBy("partition_id").count().orderBy("partition_id").show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 9375|
|           1| 9374|
|           2| 9376|
|           3| 9375|
|           4| 9375|
|           5| 9375|
|           6| 9375|
|           7| 9375|
+------------+-----+



In [38]:
shipments_12_df = shipments_df.repartition(12)
coalesced_3_df = shipments_12_df.coalesce(3)
repartitioned_3_df = shipments_12_df.repartition(3)
print("12:", shipments_12_df.rdd.getNumPartitions(), "coalesce:", coalesced_3_df.rdd.getNumPartitions(), "repartition:", repartitioned_3_df.rdd.getNumPartitions())
print("coalesce plan:"); coalesced_3_df.explain()
print("repartition plan:"); repartitioned_3_df.explain()

12: 12 coalesce: 3 repartition: 3
coalesce plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- Coalesce 3
      +- ShuffleQueryStage 0
         +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=2551]
            +- FileScan csv [shipment_id#17,booking_date#18,dispatch_date#19,promised_delivery_date#20,actual_delivery_date#21,warehouse_id#22,carrier_id#23,route_id#24,customer_segment#25,service_level#26,package_weight_kg#27,shipping_fee_inr#28,shipping_cost_inr#29,shipment_status#30,delivery_attempts#31,damage_flag#32,customer_rating#33,handling_minutes#34,payment_mode#35,priority_flag#36] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/shipments_75000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<shipment_id:string,booking_date:date,dispatch_date:date,promised_delivery_date:date,actual...
+- == Initial Plan ==
   Coalesce 3
   +- Exchange Roun

In [39]:
parquet_path = "/content/shipments_with_warehouse_parquet"
shipment_warehouse_df = shipments_df.join(warehouse_df, "warehouse_id", "left").repartition(4)
shipment_warehouse_df.write.mode("overwrite").parquet(parquet_path)
print("Saved to:", parquet_path)
print("Files:", [p.name for p in Path(parquet_path).iterdir()])

Saved to: /content/shipments_with_warehouse_parquet
Files: ['_SUCCESS', 'part-00000-228d5cdf-5acd-428a-aa74-2bc4bb888dcf-c000.snappy.parquet', 'part-00003-228d5cdf-5acd-428a-aa74-2bc4bb888dcf-c000.snappy.parquet', 'part-00002-228d5cdf-5acd-428a-aa74-2bc4bb888dcf-c000.snappy.parquet', '.part-00001-228d5cdf-5acd-428a-aa74-2bc4bb888dcf-c000.snappy.parquet.crc', '.part-00003-228d5cdf-5acd-428a-aa74-2bc4bb888dcf-c000.snappy.parquet.crc', '.part-00000-228d5cdf-5acd-428a-aa74-2bc4bb888dcf-c000.snappy.parquet.crc', '.part-00002-228d5cdf-5acd-428a-aa74-2bc4bb888dcf-c000.snappy.parquet.crc', 'part-00001-228d5cdf-5acd-428a-aa74-2bc4bb888dcf-c000.snappy.parquet', '._SUCCESS.crc']


Final Challenge

In [40]:
challenge_base_df = shipments_df.withColumn("booking_date", F.to_date("booking_date")).withColumn(
    "promised_delivery_date", F.to_date("promised_delivery_date")).withColumn(
    "actual_delivery_date", F.to_date("actual_delivery_date")).withColumn(
    "shipping_cost_clean_inr", F.coalesce(F.col("shipping_cost_inr"), F.lit(0.0))).withColumn(
    "delivery_duration_days", F.datediff("actual_delivery_date", "booking_date")).withColumn(
    "is_late", F.when(F.col("actual_delivery_date").isNull(), F.lit(None).cast("int"))
    .when(F.col("actual_delivery_date") > F.col("promised_delivery_date"), 1).otherwise(0)).withColumn(
    "shipment_profit_inr", F.col("shipping_fee_inr") - F.col("shipping_cost_clean_inr"))


In [41]:
challenge_joined_df = challenge_base_df.join(warehouse_df.select("warehouse_id", "region"), "warehouse_id", "left").join(carrier_df, "carrier_id", "left").join(route_df, "route_id", "left")


In [43]:
challenge_aggregated_df = challenge_joined_df.groupBy("region", "carrier_id").agg(
    F.sum("shipping_fee_inr").alias("total_revenue_inr"),
    F.sum("shipment_profit_inr").alias("total_profit_inr"),
    (F.avg("is_late") * 100).alias("late_delivery_pct"))


In [44]:
rank_window = Window.partitionBy("region").orderBy(F.asc("late_delivery_pct"), F.desc("total_revenue_inr"))
challenge_final_df = challenge_aggregated_df.withColumn("on_time_rank", F.dense_rank().over(rank_window)).cache()


In [45]:
challenge_final_df.orderBy("region", "on_time_rank").show(truncate=False)
print("Groups:", challenge_final_df.count())
print("Total profit:", challenge_final_df.agg(F.sum("total_profit_inr")).first()[0])


+------+----------+------------------+------------------+------------------+------------+
|region|carrier_id|total_revenue_inr |total_profit_inr  |late_delivery_pct |on_time_rank|
+------+----------+------------------+------------------+------------------+------------+
|East  |C01       |393862.61999999994|101172.6200000001 |38.40877914951989 |1           |
|East  |C04       |211583.64000000004|54990.74          |40.64171122994652 |2           |
|East  |C02       |373590.02         |96659.62000000007 |41.9170243204578  |3           |
|East  |C05       |243322.91999999998|65204.91999999997 |41.97002141327623 |4           |
|East  |C07       |222025.85999999993|53262.06          |42.18009478672986 |5           |
|East  |C03       |302260.92000000004|82083.2199999999  |43.214285714285715|6           |
|East  |C06       |195810.3399999999 |50231.34000000002 |47.82608695652174 |7           |
|East  |NULL      |24594.6           |5427.6            |50.0              |8           |
|North |C0

In [46]:
challenge_output_path = "/content/logistics_performance_report_parquet"
challenge_final_df.repartition("region").write.mode("overwrite").partitionBy("region").parquet(challenge_output_path)
print("Report saved to:", challenge_output_path)
challenge_final_df.unpersist()


Report saved to: /content/logistics_performance_report_parquet


DataFrame[region: string, carrier_id: string, total_revenue_inr: double, total_profit_inr: double, late_delivery_pct: double, on_time_rank: int]